# R1-01 · Traer el dato — Profundización (opcional) 🔬

**Formación Pública — Rama R1 · Analista de Datos · Notebook de profundización**

Opcional. **JSON anidado**, carga robusta con **descarga + caché en disco**, **normalización de
tipos** y una consulta **en vivo a la API de datos.gob.cl**. Mismos datos de matrícula escolar.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, json, os, urllib.request
CSV = "matricula.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/matricula.csv", CSV)
df = pd.read_csv(CSV)
print("Datos:", df.shape)

## 1. JSON anidado

Las respuestas reales anidan listas dentro de objetos. Practica navegarlas.

### ✍️ Ejercicio 1 — Extrae los niveles anidados

In [ ]:
payload = '{"rbd": 1, "niveles": [{"nivel": "parvularia", "matricula": 45}, {"nivel": "basica", "matricula": 320}, {"nivel": "media", "matricula": 210}]}'
data = json.loads(payload)
niveles = [n["nivel"] for n in data["niveles"]]
print(niveles)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert niveles == ["parvularia", "basica", "media"]
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "[n['nivel'] for n in data['niveles']].")
    print("   Detalle:", e)

## 2. Carga robusta con descarga y caché en disco

El patrón completo: si no está en disco, descarga; si la descarga falla, avisa sin romper.

### ✍️ Ejercicio 2 — Escribe `asegurar_csv`

In [ ]:
def asegurar_csv(ruta, url):
    if os.path.exists(ruta):
        return "local"
    try:
        urllib.request.urlretrieve(url, ruta)
        return "descargado"
    except Exception:
        return "sin_datos"

estado = asegurar_csv("matricula.csv", "http://no-existe.invalid/x.csv")
print(estado)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert estado == "local"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "El archivo ya existe localmente, así que debe devolver 'local'.")
    print("   Detalle:", e)

## 3. Normalizar tipos al cargar

Los datos vienen sucios: a veces los números llegan como texto. Conviértelos con seguridad.

### ✍️ Ejercicio 3 — Escribe `a_float`

In [ ]:
def a_float(x):
    try:
        return float(str(x).replace("$", "").replace(".", "").replace(",", "."))
    except ValueError:
        return None

print(a_float("1.234,50"), a_float("$2000"), a_float("N/A"))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert a_float("1.234,50") == 1234.50
    assert a_float("N/A") is None
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'En el except ValueError: return None.')
    print("   Detalle:", e)

## 4. Una API estatal de verdad: datos.gob.cl

El portal de datos abiertos del Estado tiene una **API real** (CKAN). Consultémosla en vivo, con caché de respaldo por si no hay red: el mismo patrón de la lección, contra una API que existe.

### ✍️ Ejercicio 4 — Consulta cuántos datasets de matrícula hay en datos.gob.cl

In [ ]:
URL = "https://datos.gob.cl/api/3/action/package_search?q=matricula&rows=3"
CACHE = '{"success": true, "result": {"count": 42, "results": [{"title": "Matrícula (caché)"}]}}'

def consultar_api(url):
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            return json.loads(r.read().decode())
    except Exception:
        return json.loads(CACHE)

data = consultar_api(URL)
n_datasets = data["result"]["count"]
print("datasets sobre matrícula en datos.gob.cl:", n_datasets)
for res in data["result"]["results"]:
    print(" -", res["title"])

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert data["success"] is True
    assert isinstance(n_datasets, int) and n_datasets > 0
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "data['result']['count'] — navega el JSON igual que en el ejercicio 1.")
    print("   Detalle:", e)

---

*Fin de la profundización de R1-01.*